In [ ]:
!pip install torch transformers spacy networkx scikit-learn pandas lxml
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import RobertaTokenizer, RobertaModel
import spacy, networkx as nx, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import xml.etree.ElementTree as ET


In [ ]:
def load_semeval_dataset(path):
    tree = ET.parse(path)
    root = tree.getroot()
    data = []
    for sentence in root.findall('sentence'):
        text = sentence.find('text').text
        for opinion in sentence.find('Opinions').findall('Opinion'):
            aspect = opinion.get('target')
            polarity = opinion.get('polarity')
            if polarity == "positive": label = 0
            elif polarity == "negative": label = 1
            else: label = 2
            data.append({"sentence": text, "aspect": aspect, "label": label})
    return pd.DataFrame(data)

def preprocess_dataset(df):
    df = df.dropna()
    df['sentence'] = df['sentence'].str.lower()
    return df

laptop_train = preprocess_dataset(load_semeval_dataset("SemEval14/Laptop_Train.xml"))
restaurant_train = preprocess_dataset(load_semeval_dataset("SemEval14/Restaurants_Train.xml"))
laptop_test = preprocess_dataset(load_semeval_dataset("SemEval14/Laptop_Test.xml"))
restaurant_test = preprocess_dataset(load_semeval_dataset("SemEval14/Restaurants_Test.xml"))

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
roberta = RobertaModel.from_pretrained("roberta-base")

def get_embeddings(sentence):
    inputs = tokenizer(sentence, return_tensors="pt")
    outputs = roberta(**inputs)
    return outputs.last_hidden_state.squeeze(0)


In [ ]:
nlp = spacy.load("en_core_web_sm")

def build_syntactic_graph(sentence):
    doc = nlp(sentence)
    G = nx.Graph()
    for token in doc:
        G.add_node(token.i, word=token.text)
        if token.head.i != token.i:
            G.add_edge(token.i, token.head.i)
    return torch.tensor(nx.to_numpy_array(G), dtype=torch.float)


In [ ]:


lexicon = pd.read_csv("NRC_emotion_lexicon.csv")

# Map sentiment to index vector [Happy, Sadness, Fear, Anger, Surprise, Disgust]
emotion_index = {"joy":0, "sadness":1, "fear":2, "anger":3, "surprise":4, "disgust":5}

# Create emotion_dict from lexicon
emotion_dict = {}
for _, row in lexicon.iterrows():
    word, emotion, assoc = row['word'], row['emotion'], row['association']
    if word not in emotion_dict:
        emotion_dict[word] = [0.0]*6
    if assoc == 1 and emotion in emotion_index:
        emotion_dict[word][emotion_index[emotion]] = 1.0

# Function get vector fuzzy for one word
def fuzzy_emotion_vector(word):
    vec = emotion_dict.get(word.lower(), [0.0]*6)
    return torch.tensor(vec, dtype=torch.float)

# Create node features (embedding + fuzzy emotion)
def build_node_features(sentence):
    emb = get_embeddings(sentence)
    doc = nlp(sentence)
    fuzzy = torch.stack([fuzzy_emotion_vector(tok.text) for tok in doc])
    return torch.cat([emb, fuzzy], dim=1)


In [ ]:
def show_sample_emotions(df, emotion_dict, n=10):
    vocab = list(set(" ".join(df['sentence']).split()))
    sample_words = vocab[:n]
    for word in sample_words:
        vec = emotion_dict.get(word, [0.0]*6)
        print(f"{word:15} -> {vec}")

print("Sample fuzzy emotion vectors (Laptop dataset):")
show_sample_emotions(laptop_train, emotion_dict, n=10)


In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_feats, out_feats):
        super().__init__()
        self.linear = nn.Linear(in_feats, out_feats)
    def forward(self, x, adj):
        h = torch.mm(adj, x)
        return F.relu(self.linear(h))

class OFEG(nn.Module):
    def __init__(self, hidden_size=768, gcn_hidden=256, num_classes=3):
        super().__init__()
        self.gcn = GCNLayer(hidden_size+6, gcn_hidden)
        self.attn = nn.MultiheadAttention(embed_dim=gcn_hidden, num_heads=4)
        self.fc = nn.Linear(gcn_hidden, num_classes)
    def forward(self, x, adj):
        h = self.gcn(x, adj)
        h = h.unsqueeze(0)
        attn_out, _ = self.attn(h, h, h)
        pooled = attn_out.mean(dim=1)
        return self.fc(pooled)


In [ ]:
from torch.utils.data import Dataset, DataLoader

class SemEvalDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data.iloc[idx]
        sentence, label = item["sentence"], item["label"]
        x = build_node_features(sentence)
        adj = build_syntactic_graph(sentence)
        return x, adj, label

train_dataset = SemEvalDataset(laptop_train)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_dataset = SemEvalDataset(laptop_test)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)


In [ ]:
model = OFEG()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    losses, preds, trues = [], [], []
    for x, adj, label in train_loader:
        output = model(x.squeeze(0), adj.squeeze(0))
        loss = criterion(output, torch.tensor([label]))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
        preds.append(torch.argmax(output).item()); trues.append(label)
    print(f"Epoch {epoch+1}, Loss={np.mean(losses):.4f}, Acc={accuracy_score(trues,preds):.2f}, F1={f1_score(trues,preds,average='macro'):.2f}")


In [ ]:
def evaluate_model(model, dataloader):
    preds, trues = [], []
    with torch.no_grad():
        for x, adj, label in dataloader:
            output = model(x.squeeze(0), adj.squeeze(0))
            preds.append(torch.argmax(output).item())
            trues.append(label)
    return accuracy_score(trues, preds), f1_score(trues, preds, average="macro")

acc, f1 = evaluate_model(model, test_loader)
print(f"Laptop Test - Accuracy: {acc:.2f}, F1: {f1:.2f}")
